# Lesson 82 · 🎨 Music features visualized — making music show its true shape: chroma heatmaps, beat grids and similarity matrices

**Goal**: draw the chroma, the beat grid and the similarity matrix, and learn to read the structure of music features across time and pitch.

> **🎨**: reading the pictures comes first; the embeddings can be demonstrated with synthetic data.

🔗 Aurora connection: this lesson is the visual display layer of the `aurora.music` deliverable, mapping to `src/aurora/music/features.py` (chroma extraction, beat tracking) and `src/aurora/music/similarity.py` (cosine-similarity heatmaps). The embeddings in this lesson are synthetic stand-ins; the real contrastive-learning embeddings are implemented in L79.

← **Previous lesson** · [L81 · A music recommender](L81_recommendation.ipynb)

> Last lesson covered **a music recommender**: user taste → embeddings → k-NN neighbors → a recommendation list.
> This lesson explores **music features visualized**.

## Learning goals

By the end of this lesson you should be able to:

1. **Read a chroma heatmap**: recognize the chroma pattern of a I-IV-V-I chord progression, and tell the pitch-class combinations of major and minor triads apart.
2. **Understand t-SNE's hyperparameters**: explain how low/high `perplexity` affects the clustering (shattering vs merging), and know what a plot looks like when `n_iter` is too small to converge.
3. **Quantify style distance with cosine similarity**: read the expected "same style > across styles" pattern off a 4×4 similarity matrix, and explain mathematically why the diagonal is always 1.0.
4. **Recognize a beat grid**: understand how the onset envelope relates to beat_frames, and the formula converting BPM → frame spacing.

The core question of music understanding is: **how "alike" are two pieces of audio?**
Chroma folds the spectrum into 12 pitch bins, so chord structure is visible at a glance;
an embedding squeezes a whole piece into one vector, so "alike" becomes a distance between vectors;
t-SNE draws those vectors on a 2D plane, so you can check whether the model really learned musical similarity.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from sklearn.manifold import TSNE
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print("⚠️ sklearn is not installed; the t-SNE visualizations will be skipped")

## 1. A chroma piano roll

Chroma (the chroma feature) folds every frequency into 12 bins (C, C#, D, … B) using the **octave equivalence** of pitch.
Formula: the pitch class of frequency `f` is `p = round(12 * log2(f / 440) + 69) mod 12`.

*Here +69 is the MIDI number of A4 (440 Hz), and mod 12 normalizes C to 0. Check: plug in A4 → round(0 + 69) mod 12 = 69 mod 12 = 9 (A) ✓*

How to read the plot: horizontal axis = time frame, vertical axis = the 12 pitches (0=C … 11=B), brightness = the energy of that pitch in that frame.
Reading tip: C(0) + E(4) + G(7) lit up at the same moment → a major triad; C(0) + Eb(3) + G(7) → a minor triad.

In [ ]:
# Simulate the chroma matrix of a "C major I-IV-V-I" chord progression
# Each chord lasts 16 frames: C major (0,4,7) → F major (5,9,0) → G major (7,11,2) → C major (0,4,7)
rng = np.random.default_rng(0)  # fixed seed so the noise texture is reproducible
T = 64  # total number of frames
chroma = np.zeros((12, T))

chord_sequence = [
    ([0, 4, 7], 0, 16),    # C major triad
    ([5, 9, 0], 16, 32),   # F major triad
    ([7, 11, 2], 32, 48),  # G major triad
    ([0, 4, 7], 48, 64),   # C major triad (return home)
]

for notes, t_start, t_end in chord_sequence:
    for n in notes:
        chroma[n, t_start:t_end] = 1.0 + 0.15 * rng.standard_normal(t_end - t_start)

chroma = np.clip(chroma + 0.05 * np.abs(rng.standard_normal(chroma.shape)), 0, 1.5)

pitch_names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(chroma, aspect='auto', origin='lower', cmap='magma',
               vmin=0, vmax=1.5, interpolation='nearest')
ax.set_yticks(range(12))
ax.set_yticklabels(pitch_names, fontsize=9)
ax.set_xlabel('Time frame')
ax.set_ylabel('Pitch class')
ax.set_title('Chroma piano roll — I-IV-V-I chord progression in C major')
for label, t_mid in [('C', 8), ('F', 24), ('G', 40), ('C', 56)]:
    ax.text(t_mid, 12.2, label, ha='center', va='bottom',
            color='white', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, label='Energy')
plt.tight_layout()
plt.show()
print('✅ Chroma plot: vertical bright bars = chord notes active together; read the progression left to right')

In [ ]:
# Check the shape of the music features
assert chroma.shape[0] == 12, f"the chroma feature should have 12 pitch classes, got {chroma.shape[0]}"
assert chroma.shape[1] > 0, f"the number of time frames should be > 0, got {chroma.shape[1]}"
print(f"✅ chroma shape: {chroma.shape}  (12 pitch classes × {chroma.shape[1]} time frames)")

## 2. Embeddings and t-SNE dimensionality reduction

Contrastive learning encodes each piece of music as a `d`-dimensional vector, with the goal of keeping vectors of music from the **same** class close together and those from **different** classes far apart.
Loss function (InfoNCE): `L = -log( exp(sim(q,k+)/τ) / Σ exp(sim(q,ki)/τ) )`, where `sim` is cosine similarity and `τ` is the temperature hyperparameter.

t-SNE reduces high-dimensional vectors to 2D while preserving local neighborhood structure:
it computes a probability distribution `p_ij` over pairwise similarities in the high-dimensional space, matches it with a Student-t distribution `q_ij` in the low-dimensional space, and minimizes the KL divergence `KL(P || Q)`.
The `perplexity` parameter (usually 5–50) controls the neighborhood size: the larger it is, the more neighbors are taken into account.

In [ ]:
# Simulation: 4 music styles, 40 pieces each, vector dimension 64
np.random.seed(42)
n_per_class = 40
dim = 64
genres = ['Classical', 'Jazz', 'Rock', 'Electronic']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

# "Trained" embeddings: each class has a clear center and small within-class variance
centers_trained = np.array([
    np.random.randn(dim) * 3 + np.array([10, 0] + [0] * (dim - 2)),
    np.random.randn(dim) * 3 + np.array([0, 10] + [0] * (dim - 2)),
    np.random.randn(dim) * 3 + np.array([-10, 0] + [0] * (dim - 2)),
    np.random.randn(dim) * 3 + np.array([0, -10] + [0] * (dim - 2)),
])
X_trained = np.vstack([
    centers_trained[i] + np.random.randn(n_per_class, dim) * 1.5
    for i in range(4)
])

# "Untrained" embeddings: random initialization, no structure
X_random = np.random.randn(n_per_class * 4, dim)

labels = np.repeat(np.arange(4), n_per_class)

if HAS_SKLEARN:
    # t-SNE dimensionality reduction
    tsne = TSNE(n_components=2, perplexity=20, random_state=0, max_iter=1000)
    Z_trained = tsne.fit_transform(X_trained)
    Z_random  = tsne.fit_transform(X_random)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, Z, title in zip(
        axes,
        [Z_random, Z_trained],
        ['Random initialization (no structure)', 'After contrastive learning (classes cluster)']
    ):
        for i, (genre, c) in enumerate(zip(genres, colors)):
            mask = labels == i
            ax.scatter(Z[mask, 0], Z[mask, 1], c=c, label=genre,
                       alpha=0.75, s=40, edgecolors='none')
        ax.set_title(title, fontsize=12)
        ax.legend(loc='best', fontsize=9)
        ax.set_xlabel('t-SNE dim 1')
        ax.set_ylabel('t-SNE dim 2')
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle('t-SNE: music embedding clusters compared', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
    print('✅ Left: random vectors scattered with no structure; right: after contrastive learning, each class gathers into a clump')
else:
    print("⚠️ t-SNE visualization skipped (sklearn is not installed)")

## 3. Parameter experiment: how `perplexity` and `n_iter` affect t-SNE

| Parameter | Low value | High value | What you should see |
|------|------|------|----------|
| `perplexity` | 5 | 50 | low → clusters shatter into small balls; high → clusters stretch and merge, global structure is clearer |
| `n_iter` | 250 | 2000 | too few iterations → the shape hasn't converged, cluster boundaries are blurry; enough → stable separation |
| `dim` (input dimension) | 8 | 256 | high-dimensional with no contrastive learning → random embeddings are harder to separate |

Suggested experiment: adjust `perplexity` and `n_iter` below (the number of iterations; from sklearn 1.5 on this parameter is renamed `max_iter`, and the code already uses `max_iter`), and watch whether the four kinds of music stay separated.

In [ ]:
if HAS_SKLEARN:
    # Parameter experiment: vary perplexity and compare the clustering
    perplexity_list = [5, 20, 50]  # ← change this
    n_iter_exp = 500               # ← you can change this too (250 / 500 / 2000)

    fig, axes = plt.subplots(1, len(perplexity_list),
                             figsize=(5 * len(perplexity_list), 4))
    for ax, perp in zip(axes, perplexity_list):
        tsne_exp = TSNE(n_components=2, perplexity=perp,
                        random_state=0, max_iter=n_iter_exp)
        Z_exp = tsne_exp.fit_transform(X_trained)
        for i, (genre, c) in enumerate(zip(genres, colors)):
            mask = labels == i
            ax.scatter(Z_exp[mask, 0], Z_exp[mask, 1], c=c, label=genre,
                       alpha=0.8, s=35, edgecolors='none')
        ax.set_title(f'perplexity={perp}\nn_iter={n_iter_exp}', fontsize=10)
        ax.legend(fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle('t-SNE perplexity experiment (embeddings after contrastive learning)', y=1.02)
    plt.tight_layout()
    plt.show()

    print(f'✅ Experiment done: perplexity={perplexity_list}, n_iter={n_iter_exp}')
    print('Expected: perplexity=5 shatters the clusters; perplexity=50 gives smoother cluster boundaries but may stretch them')
else:
    print("⚠️ t-SNE parameter experiment skipped (sklearn is not installed)")

## 4. Visualizing the beat grid

**How it works**: an onset-detection algorithm looks for frames where the energy jumps in the time domain (strong onsets), then uses autocorrelation to estimate the main period → BPM. The result is drawn as an onset envelope (line) + a beat grid (vertical lines).

Below we demonstrate with a synthetic drum signal: `BPM=120`, each beat produces a spike in the onset envelope, and red vertical lines mark the beat positions.

In [ ]:
# Synthetic beat envelope: BPM=120, 4 seconds
sr, hop = 22050, 512
bpm, duration = 120.0, 4.0
n_frames = int(duration * sr / hop)
t = np.arange(n_frames) * hop / sr

rng = np.random.default_rng(0)
onset_env = rng.random(n_frames) * 0.08   # background noise

# Boost an onset spike at every beat position (main peak + two decaying echoes)
beat_hop = sr / (bpm / 60) / hop          # beat spacing (in frames)
beat_frames = np.arange(0, n_frames, beat_hop).astype(int)
for bf in beat_frames:
    for offset, gain in [(0, 1.0), (1, 0.4), (2, 0.1)]:
        idx = bf + offset
        if 0 <= idx < n_frames:
            onset_env[idx] += gain

fig, ax = plt.subplots(figsize=(12, 3))
ax.fill_between(t, onset_env, alpha=0.35, color='#2A9D8F')
ax.plot(t, onset_env, color='#2A9D8F', lw=1, label='Onset envelope')
for bf in beat_frames:
    if bf < n_frames:
        ax.axvline(t[bf], color='#E76F51', lw=1.5, alpha=0.7)
ax.axvline(t[beat_frames[0]], color='#E76F51', lw=1.5, alpha=0.7,
           label=f'Beat grid ({int(bpm)} BPM)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Onset strength')
ax.set_title(f'Beat grid visualized — BPM={int(bpm)}, beat spacing {60/bpm:.3f} s')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

expected = int(duration * bpm / 60)
print(f'Duration {duration} s × {bpm} BPM → {expected} beats expected, {len(beat_frames)} peaks detected')
print(f'Beat spacing: {60/bpm:.3f} s (= {beat_hop:.1f} frames × {hop} hop)')

## 5. Similarity heatmap

Use each style's **mean** embedding as its representative vector and compute the 4×4 cosine-similarity matrix.
- Diagonal = 1.0 (self-similarity)
- When contrastive learning succeeds: same style > 0.9, across styles < 0.3

In [ ]:
# Use each style's average embedding as its representative vector
genre_means = np.vstack([
    X_trained[labels == i].mean(axis=0) for i in range(4)
])
norms = np.linalg.norm(genre_means, axis=1, keepdims=True) + 1e-9
sim = (genre_means / norms) @ (genre_means / norms).T   # (4, 4)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim, vmin=-1, vmax=1, cmap='RdBu_r', aspect='equal')
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_xticks(range(4));  ax.set_yticks(range(4))
ax.set_xticklabels(genres, fontsize=10)
ax.set_yticklabels(genres, fontsize=10)
for i in range(4):
    for j in range(4):
        col = 'white' if abs(sim[i, j]) > 0.6 else 'black'
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                fontsize=10, color=col)
ax.set_title('Style similarity heatmap (cosine similarity)')
plt.tight_layout()
plt.show()

same = [sim[i, i] for i in range(4)]
cross = [sim[i, j] for i in range(4) for j in range(4) if i != j]
print(f'Diagonal (self): mean {sum(same)/len(same):.3f}')
print(f'Across styles: mean {sum(cross)/len(cross):.3f} (smaller → better style separation)')

## Exercise: spotting a minor triad

The pitch-class combination of a minor triad (relative to its root) is `[0, 3, 7]` (root, minor third, perfect fifth).
In a chroma matrix, a time frame that has "activated" a minor triad means those three pitch classes all have fairly high energy at once.

**Task**: implement the function `detect_minor_triad(chroma_frame, threshold)`.
Given one frame of chroma (shape `(12,)`), return the list of all possible roots `r` (0–11)
for which the three pitches `(r, (r+3)%12, (r+7)%12)` all have energy above `threshold`.

In [ ]:
def detect_minor_triad(chroma_frame: np.ndarray, threshold: float = 0.5) -> list[int]:
    """
    Detect minor-triad roots in one frame of chroma.

    Args:
        chroma_frame: shape (12,), the normalized energy (0–1) of each pitch class.
        threshold: energy threshold; above it a pitch counts as "active".

    Returns:
        List of all roots (0–11) that satisfy the condition; an empty list means no minor triad was detected.
    """
    # TODO: loop over the 12 possible roots r and check whether (r, (r+3)%12, (r+7)%12) all exceed threshold
    raise NotImplementedError("Please implement detect_minor_triad")


# ── Verification ──────────────────────────────────────────────────────────────────────
try:
    # Synthesize one frame: A minor triad A(9) C(0) E(4) → root=9, 9+3=0(mod12), 9+7=4(mod12)
    test_frame = np.zeros(12)
    test_frame[[9, 0, 4]] = 0.8  # A minor triad
    test_frame[2] = 0.3           # noise

    roots = detect_minor_triad(test_frame, threshold=0.5)
    assert 9 in roots, f"should detect root 9 (A), got {roots}"
    assert 0 not in roots or test_frame[0] > 0.5, "C should not be mistaken for a root (unless C/Eb/G all light up)"
    print(f"✅ Minor-triad roots detected: {[['C','C#','D','D#','E','F','F#','G','G#','A','A#','B'][r] for r in roots]}")
except (NotImplementedError, TypeError):
    print("⚠️ Not implemented yet — fill in the body of detect_minor_triad")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")

## Wrap-up

This lesson used a `chroma` matrix to visualize the time–pitch structure of a chord progression, so the 12-row heatmap let you "read off" the I-IV-V-I progression directly; it used t-SNE to squeeze 64-dimensional embeddings down to 2D and compared the clustering of random initialization with that after contrastive learning; a synthetic beat envelope demonstrated onset detection and beat-grid marking; and a 4×4 cosine-similarity heatmap showed the distances between styles at a glance. Together the four plots form the core debugging interface of the `aurora.music` module (`features.py`: chroma and beat tracking; `similarity.py`: the cosine-similarity matrix).

Next lesson: **L83 — a Transformer from scratch**: implement multi-head attention and positional encoding, and walk the core derivation of the attention mechanism from maths to code.

In [ ]:
# ── Standalone maths assertions: core properties of music features (no student code needed) ─────────────────────────────

# 1. Interval structure of a minor triad: root r → (r, r+3, r+7) mod 12
for root in range(12):
    minor_triad = [root, (root+3)%12, (root+7)%12]
    interval_3rd = (minor_triad[1] - minor_triad[0]) % 12
    interval_5th = (minor_triad[2] - minor_triad[0]) % 12
    assert interval_3rd == 3, f"root={root}: a minor third should be 3 semitones, got {interval_3rd}"
    assert interval_5th == 7, f"root={root}: a perfect fifth should be 7 semitones, got {interval_5th}"
print("1 ✅  all 12 minor triads: root + (minor third = 3) + (perfect fifth = 7), holds mod 12")

# 2. In the I-IV-V-I chroma matrix, the chord tones should carry more energy than the background
# The C major triad (0,4,7) should be active in frame 0
c_major_notes = [0, 4, 7]
other_notes = [n for n in range(12) if n not in c_major_notes]
assert chroma[c_major_notes, 0].mean() > chroma[other_notes, 0].mean(), \
    f"C major chord in frame 0: mean chord-tone energy {chroma[c_major_notes,0].mean():.3f} should be > background {chroma[other_notes,0].mean():.3f}"
print(f"2 ✅  C major chord in frame 0: chord-tone energy={chroma[c_major_notes,0].mean():.3f} > background={chroma[other_notes,0].mean():.3f}")

# 3. The similarity matrix diagonal = 1 (a vector's cosine similarity with itself = 1)
for i in range(4):
    assert abs(sim[i, i] - 1.0) < 1e-9, f"sim[{i},{i}] should be 1.0, got {sim[i,i]:.6f}"
print(f"3 ✅  the similarity matrix diagonal is all 1.0 (after L2 normalization, self dot product = ||v||²/||v||² = 1)")

# 4. Check the beat-grid spacing: beat_hop = sr / (bpm/60) / hop
expected_beat_hop = sr / (bpm / 60) / hop
actual_beat_hop = (beat_frames[1] - beat_frames[0]) if len(beat_frames) > 1 else expected_beat_hop
assert abs(actual_beat_hop - expected_beat_hop) < 1.0, \
    f"beat spacing should be {expected_beat_hop:.2f} frames, got {actual_beat_hop:.2f}"
print(f"4 ✅  beat spacing={actual_beat_hop:.1f} frames = sr/{bpm/60:.0f}/hop = {expected_beat_hop:.1f} frames (BPM={int(bpm)})")

# 5. BPM → time interval round trip
beat_period_sec = 60.0 / bpm
beat_period_frames_calc = beat_period_sec * sr / hop
bpm_recovered = 60.0 / (beat_period_frames_calc * hop / sr)
assert abs(bpm_recovered - bpm) < 1e-9, f"BPM round-trip error={abs(bpm_recovered-bpm):.2e}"
print(f"5 ✅  BPM round trip: {bpm}→{beat_period_sec:.3f}s→{beat_period_frames_calc:.1f} frames→{bpm_recovered:.1f} BPM (lossless)")


---

→ **Next lesson** · [L83 · A Transformer from scratch](../9_llm/L83_transformer.ipynb)

> Next lesson covers **a Transformer from scratch**: a complete implementation of multi-head attention + positional encoding + feed-forward.